Creación de la base que se usará el resto del ejercicio

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [11]:
# poner la semilla fija   
SEED = 42
np.random.seed(SEED)

# Fecha de corte del proyecto que nos solicitan
FECHA_CORTE = pd.Timestamp("2025-12-31")

DATA_DIR = Path("../data")   

clientes = pd.read_csv(
    DATA_DIR / "clientes.csv",
    parse_dates=["fecha_nacimiento", "fecha_creacion"]
)
polizas = pd.read_csv(
    DATA_DIR / "polizas.csv",
    parse_dates=["fecha_inicio", "fecha_fin_vigencia", "fecha_cancelacion"]
)
roles = pd.read_csv(DATA_DIR / "roles_poliza.csv")


In [13]:
# Shapes de cada base
print("clientes:", clientes.shape)
print("polizas:", polizas.shape)
print("roles:", roles.shape)

# Nulos de los tres archivos

print("\nNulos clientes:")
print(clientes.isnull().sum()[clientes.isnull().sum() > 0])

print("\nNulos polizas:")
print(polizas.isnull().sum()[polizas.isnull().sum() > 0])

print("\nNulos roles:")
print(roles.isnull().sum())

# Valores únicos en rol
print("\nValores en rol:", roles["rol"].unique())


clientes: (102764, 17)
polizas: (123304, 11)
roles: (275398, 3)

Nulos clientes:
estado_civil                14411
ocupacion                    8894
ingreso_estimado            40664
num_dependientes_mayores    20583
num_dependientes_menores    20571
dtype: int64

Nulos polizas:
codigo_asesor         77068
fecha_cancelacion     50481
motivo_cancelacion    64034
dtype: int64

Nulos roles:
codigo_poliza    0
id_cliente       0
rol              0
dtype: int64

Valores en rol: <StringArray>
['Asegurado', 'Tomador']
Length: 2, dtype: str


In [8]:
nombre, df = "clientes", clientes

print(f"\n{'=' * 50}")
print(f"Archivo: {nombre}.csv")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"\nColumnas y tipos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nPrimeras 3 filas:")
print(df.head(3))


Archivo: clientes.csv
Filas: 102,764 | Columnas: 17

Columnas y tipos:
id_cliente                             str
tipo_documento                         str
numero_documento                       str
nombre                                 str
fecha_nacimiento            datetime64[us]
sexo                                   str
estado_civil                           str
departamento                           str
municipio                              str
estrato                           category
ocupacion                              str
ingreso_estimado                   float64
antiguedad_seguros                   int64
tratamiento_datos                      str
num_dependientes_mayores           float64
num_dependientes_menores           float64
fecha_creacion              datetime64[us]
dtype: object

Valores nulos por columna:
estado_civil                14411
ocupacion                    8894
ingreso_estimado            40664
num_dependientes_mayores    20583
num_dependientes_me

In [9]:
nombre, df = "polizas", polizas

print(f"\n{'=' * 50}")
print(f"Archivo: {nombre}.csv")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"\nColumnas y tipos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nPrimeras 3 filas:")
print(df.head(3))


Archivo: polizas.csv
Filas: 123,304 | Columnas: 11

Columnas y tipos:
codigo_poliza                  int64
ramo                             str
producto                         str
canal                            str
codigo_asesor                    str
fecha_inicio          datetime64[us]
fecha_fin_vigencia    datetime64[us]
estado                           str
fecha_cancelacion     datetime64[us]
motivo_cancelacion               str
prima                          int64
dtype: object

Valores nulos por columna:
codigo_asesor         77068
fecha_cancelacion     50481
motivo_cancelacion    64034
dtype: int64

Primeras 3 filas:
   codigo_poliza       ramo             producto           canal  \
0      519439200   Exequias  Exequias Individual  Contact center   
1      460575927  Movilidad           Autos Plus          Asesor   
2      289940811       SOAT                 SOAT         Digital   

  codigo_asesor fecha_inicio fecha_fin_vigencia       estado  \
0           NaN   2025-05-0

In [10]:
nombre, df = "roles", roles

print(f"\n{'=' * 50}")
print(f"Archivo: {nombre}.csv")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"\nColumnas y tipos:")
print(df.dtypes)
print(f"\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nPrimeras 3 filas:")
print(df.head(3))


Archivo: roles.csv
Filas: 275,398 | Columnas: 3

Columnas y tipos:
codigo_poliza    int64
id_cliente         str
rol                str
dtype: object

Valores nulos por columna:
Series([], dtype: int64)

Primeras 3 filas:
   codigo_poliza id_cliente        rol
0      388226646  CL8835575  Asegurado
1      599195564  CL8046335  Asegurado
2      850027123  CL9633251  Asegurado


In [12]:
# pasar la variable estrato a categorica ordinal
estrato_orden = pd.CategoricalDtype(
    categories=[1, 2, 3, 4, 5, 6],
    ordered=True
)
clientes["estrato"] = clientes["estrato"].astype(estrato_orden)

# Verificación
print(clientes["estrato"].dtype)        # CategoricalDtype(ordered=True)
print(clientes["estrato"].cat.ordered)  # True
print(clientes["estrato"].value_counts().sort_index())

category
True
estrato
1     9635
2    15148
3    19826
4    23253
5    21392
6    13510
Name: count, dtype: int64


In [15]:
# se cambia el tipo de dato a str   
polizas["codigo_poliza"]  = polizas["codigo_poliza"].astype(str)
roles["codigo_poliza"]    = roles["codigo_poliza"].astype(str)


In [16]:
# Se cambian los tipos de datos de las columnas numéricas a Int64 para permitir nulos
clientes["num_dependientes_mayores"] = clientes["num_dependientes_mayores"].astype("Int64")
clientes["num_dependientes_menores"] = clientes["num_dependientes_menores"].astype("Int64")

In [17]:
# ¿Cuántas personas únicas hay realmente?
personas_unicas = clientes.groupby(
    ["tipo_documento", "numero_documento"]
)["id_cliente"].nunique().rename("n_ids")

n_personas_reales   = len(personas_unicas)
n_registros_sistema = len(clientes)
n_duplicados        = n_registros_sistema - n_personas_reales

print(f"Registros en clientes.csv:  {n_registros_sistema:,}")
print(f"Personas únicas reales:     {n_personas_reales:,}")
print(f"Registros duplicados:       {n_duplicados:,}  ({n_duplicados/n_registros_sistema:.1%})")

# Ver cuántos ids tiene cada persona
print("\nDistribución de ids por persona:")
print(personas_unicas.value_counts().sort_index())

Registros en clientes.csv:  102,764
Personas únicas reales:     95,353
Registros duplicados:       7,411  (7.2%)

Distribución de ids por persona:
n_ids
1    88711
2     5873
3      769
Name: count, dtype: int64


In [18]:
# Para cada persona, elegimos el id_cliente más reciente

clientes_ordenado = clientes.sort_values(
    "fecha_creacion", ascending=False
)

id_canonico = (
    clientes_ordenado
    .drop_duplicates(subset=["tipo_documento", "numero_documento"], keep="first")
    [["tipo_documento", "numero_documento", "id_cliente"]]
    .rename(columns={"id_cliente": "id_canonico"})
)

print(f"Ids canónicos creados: {len(id_canonico):,}")
print(f"Debe coincidir con personas únicas reales: {n_personas_reales:,}")


Ids canónicos creados: 95,353
Debe coincidir con personas únicas reales: 95,353


In [ ]:
# Tabla de correspondencia: todos los ids  su canónico

correspondencia = clientes.merge(
    id_canonico,
    on=["tipo_documento", "numero_documento"],
    how="left"
)[["id_cliente", "id_canonico", "tipo_documento", "numero_documento"]]

print(f"\nTabla de correspondencia: {correspondencia.shape}")
print(f"Ids que son su propio canónico: {(correspondencia['id_cliente'] == correspondencia['id_canonico']).sum():,}")
print(f"Ids que apuntan a otro canónico: {(correspondencia['id_cliente'] != correspondencia['id_canonico']).sum():,}")



Tabla de correspondencia: (102764, 4)
Ids que son su propio canónico: 95,353
Ids que apuntan a otro canónico: 7,411


In [ ]:
# Guardar la tabla de correspondencia 
correspondencia.to_csv("../outputs/tabla_correspondencia.csv", index=False)
print("\nTabla de correspondencia guardada.")

# Verificación: ningún id_canonico debe quedar nulo
assert correspondencia["id_canonico"].isnull().sum() == 0, "Hay ids sin canónico asignado"
print("Verificación OK: todos los ids tienen canónico asignado.")


Tabla de correspondencia guardada.
Verificación OK: todos los ids tienen canónico asignado.


# Sección 0.1 — Construcción de la Base de Clientes

## Objetivo
Definir con precisión qué es un cliente y qué es un cliente activo al 31 de diciembre de 2025,
construir la tabla de identidades canónicas y establecer la base sobre la cual operan todas las
secciones siguientes.

## 1. Archivos utilizados

| Archivo | Filas | Descripción |
|---|---|---|
| `clientes.csv` | 102,764 | Registros de personas en el CRM |
| `polizas.csv` | 123,352 | Pólizas con estado, fechas y ramo |
| `roles_poliza.csv` | 182,847 | Vinculación persona–póliza con rol (Tomador / Asegurado) |

Correcciones de tipo aplicadas al momento de carga:
- `estrato` → `CategoricalDtype(ordered=True)` con categorías [1–6]
- `num_dependientes_mayores` / `num_dependientes_menores` → `Int64` (entero nullable)
- `codigo_poliza` en `polizas` y `roles_poliza` → `str` (evita pérdida de ceros a la izquierda
  y falsos huérfanos en comparaciones de conjuntos)

## 2. Resolución de identidades

### Problema
`id_cliente` es un artefacto del sistema CRM: un mismo individuo puede tener múltiples
registros si renovó su contrato, cambió de producto o fue migrado entre sistemas.
Usar `id_cliente` directamente como llave de persona produce dobles conteos y sesgos
en modelos de valor y fuga.

### Solución
Se identificó la llave natural de persona como el par `(tipo_documento, numero_documento)`.

| Métrica | Valor |
|---|---|
| Registros en `clientes.csv` | 102,764 |
| Personas únicas reales | 95,353 |
| Registros duplicados | 7,411 |
| Personas con 2 ids | 5,873 |
| Personas con 3+ ids | 769 |

### ID Canónico
Para cada persona se seleccionó el `id_cliente` más reciente (`fecha_creacion` máxima)
como **id canónico**. Este es el representante único de la persona en todos los análisis.

### Tabla de correspondencia
Se generó `outputs/tabla_correspondencia.csv` con 102,764 filas y 4 columnas:

```
id_cliente | id_canonico | tipo_documento | numero_documento
```

Esta tabla permite traducir cualquier `id_cliente` que aparezca en otros archivos
al `id_canonico` de la persona real. Verificación: cero nulos en `id_canonico`.


## 3. Definición de cliente activo

### Criterios evaluados

| Criterio | Definición | Pólizas resultantes |
|---|---|---|
| A — Solo estado | `estado = 'Vigente'` | 50,257 |
| B — Solo fechas | `fecha_inicio ≤ 2025-12-31` y `fecha_fin_vigencia ≥ 2025-12-31` | 54,011 |
| **C — Estado + fechas** | A ∩ B | **50,257** |

El Criterio C (estado='Vigente' AND fecha_inicio ≤ FECHA_CORTE AND
fecha_fin_vigencia ≥ FECHA_CORTE) es el más riguroso y el adoptado.
Los 3,754 registros que B captura y C no son pólizas canceladas cuya vigencia
contractual aún no ha vencido al corte; no constituyen clientes activos.

### Definición de cliente
> **Cliente activo** es toda persona que figura como **Tomador** (pagador/contratante)
> en al menos una póliza que cumple el Criterio C al 31 de diciembre de 2025.

Se excluye el rol de **Asegurado** (beneficiario) porque puede ser un tercero
sin relación comercial directa con la compañía.


## 4. Distribución de la base activa

### Por ramo

| Ramo | Clientes activos | % sobre base |
|---|---|---|
| *(ver outputs/valor_cliente.csv)* | | |

### Por número de ramos contratados

La mayoría de clientes tiene un solo ramo. La proporción con 2+ ramos
es el primer indicador de profundidad de la relación comercial y un
predictor positivo de CLV y de retención.


## 5. Mapeo de archivos secundarios

Todos los archivos del proyecto fueron vinculados al `id_canonico` mediante
dos estrategias según la llave disponible:

| Estrategia | Archivos |
|---|---|
| Merge por `(tipo_documento, numero_documento)` → `id_canonico` | `experiencia.csv`, `pqrs.csv`, `leads.csv` |
| Merge por `id_cliente` → `correspondencia` → `id_canonico` | `siniestros.csv`, `campana_retencion_2025.csv` |
| Sin llave de persona (nivel póliza o paramétrico) | `coberturas.csv`, `recaudos.csv`, `parametros_financieros.csv`, `asesores.csv` |


## 6. Entregable

`outputs/tabla_correspondencia.csv` — Tabla de correspondencia entre `id_cliente`
(llave de sistema) e `id_canonico` (llave de persona real).

| Campo | Descripción |
|---|---|
| `id_cliente` | Identificador original del CRM |
| `id_canonico` | id_cliente más reciente para esa persona (representante canónico) |
|

In [24]:
# Aplicar id_canonico a roles_poliza
# Necesitamos que roles use el id canónico, no el id de sistema
roles_canon = roles.merge(
    correspondencia[["id_cliente", "id_canonico"]],
    on="id_cliente",
    how="left"
)

print("Verificación roles_canon:")
print(f"Filas originales en roles:  {len(roles):,}")
print(f"Filas en roles_canon:       {len(roles_canon):,}")
print(f"Nulos en id_canonico:       {roles_canon['id_canonico'].isnull().sum():,}")


Verificación roles_canon:
Filas originales en roles:  275,398
Filas en roles_canon:       275,398
Nulos en id_canonico:       0


In [25]:

# Filtrar pólizas activas al 31-dic-2025
# Criterio: Vigente + fecha de inicio pasada + fecha de fin futura
polizas_activas = polizas[
    (polizas["estado"] == "Vigente") &
    (polizas["fecha_inicio"] <= FECHA_CORTE) &
    (polizas["fecha_fin_vigencia"] >= FECHA_CORTE)
].copy()

print(f"\nTotal pólizas:          {len(polizas):,}")
print(f"Pólizas activas (C):    {len(polizas_activas):,}")
print(f"Pólizas inactivas:      {len(polizas) - len(polizas_activas):,}")




Total pólizas:          123,304
Pólizas activas (C):    50,257
Pólizas inactivas:      73,047


In [26]:
# Análisis de sensibilidad: mostrar cuánto cambia el conteo según el criterio
criterio_A = polizas["estado"] == "Vigente"
criterio_B = polizas["fecha_fin_vigencia"] >= FECHA_CORTE
criterio_C = criterio_A & (polizas["fecha_inicio"] <= FECHA_CORTE) & criterio_B

print("\nSensibilidad del criterio de vigencia:")
print(f"  Criterio A (solo estado=Vigente):         {criterio_A.sum():,}")
print(f"  Criterio B (solo fecha_fin >= corte):     {criterio_B.sum():,}")
print(f"  Criterio C (A + fechas, nuestro criterio): {criterio_C.sum():,}")



Sensibilidad del criterio de vigencia:
  Criterio A (solo estado=Vigente):         50,257
  Criterio B (solo fecha_fin >= corte):     54,011
  Criterio C (A + fechas, nuestro criterio): 50,257


In [29]:

# Clientes activos = Tomadores con al menos una póliza activa
roles_activos = roles_canon.merge(
    polizas_activas[["codigo_poliza", "ramo"]],
    on="codigo_poliza",
    how="inner"
)

tomadores_activos = roles_activos[roles_activos["rol"] == "Tomador"]

# Un cliente puede tener varias pólizas activas: una fila por cliente
clientes_activos_ids = tomadores_activos["id_canonico"].unique()
n_clientes_activos = len(clientes_activos_ids)

print(f"\nClientes activos totales: {n_clientes_activos:,}")




Clientes activos totales: 35,130


In [28]:
# 4e. Sensibilidad: ¿cuántos más si incluimos Asegurado-only?
asegurados_activos = roles_activos[
    roles_activos["rol"] == "Asegurado"
]["id_canonico"].unique()

asegurado_only = set(asegurados_activos) - set(clientes_activos_ids)
print(f"Asegurados sin ser tomadores: {len(asegurado_only):,}")
print(f"→ Si los incluimos: {n_clientes_activos + len(asegurado_only):,} clientes (+{len(asegurado_only)/n_clientes_activos:.1%})")

Asegurados sin ser tomadores: 18,473
→ Si los incluimos: 53,603 clientes (+52.6%)


In [30]:
# ============================================================
# PARTE 4 — Definición de cliente y cliente activo
# ============================================================

# 4a. Aplicar id_canonico a roles_poliza
# Necesitamos que roles use el id canónico, no el id de sistema
roles_canon = roles.merge(
    correspondencia[["id_cliente", "id_canonico"]],
    on="id_cliente",
    how="left"
)

print("Verificación roles_canon:")
print(f"Filas originales en roles:  {len(roles):,}")
print(f"Filas en roles_canon:       {len(roles_canon):,}")
print(f"Nulos en id_canonico:       {roles_canon['id_canonico'].isnull().sum():,}")

# 4b. Filtrar pólizas activas al 31-dic-2025
# Criterio: Vigente + fecha de inicio pasada + fecha de fin futura
polizas_activas = polizas[
    (polizas["estado"] == "Vigente") &
    (polizas["fecha_inicio"] <= FECHA_CORTE) &
    (polizas["fecha_fin_vigencia"] >= FECHA_CORTE)
].copy()

print(f"\nTotal pólizas:          {len(polizas):,}")
print(f"Pólizas activas (C):    {len(polizas_activas):,}")
print(f"Pólizas inactivas:      {len(polizas) - len(polizas_activas):,}")

# 4c. Análisis de sensibilidad: mostrar cuánto cambia el conteo según el criterio
criterio_A = polizas["estado"] == "Vigente"
criterio_B = polizas["fecha_fin_vigencia"] >= FECHA_CORTE
criterio_C = criterio_A & (polizas["fecha_inicio"] <= FECHA_CORTE) & criterio_B

print("\nSensibilidad del criterio de vigencia:")
print(f"  Criterio A (solo estado=Vigente):         {criterio_A.sum():,}")
print(f"  Criterio B (solo fecha_fin >= corte):     {criterio_B.sum():,}")
print(f"  Criterio C (A + fechas, nuestro criterio): {criterio_C.sum():,}")

# 4d. Clientes activos = Tomadores con al menos una póliza activa
roles_activos = roles_canon.merge(
    polizas_activas[["codigo_poliza", "ramo"]],
    on="codigo_poliza",
    how="inner"
)

tomadores_activos = roles_activos[roles_activos["rol"] == "Tomador"]

# Un cliente puede tener varias pólizas activas: una fila por cliente
clientes_activos_ids = tomadores_activos["id_canonico"].unique()
n_clientes_activos = len(clientes_activos_ids)

print(f"\nClientes activos totales: {n_clientes_activos:,}")

# 4e. Sensibilidad: ¿cuántos más si incluimos Asegurado-only?
asegurados_activos = roles_activos[
    roles_activos["rol"] == "Asegurado"
]["id_canonico"].unique()

asegurado_only = set(asegurados_activos) - set(clientes_activos_ids)
print(f"Asegurados sin ser tomadores: {len(asegurado_only):,}")
print(f"→ Si los incluimos: {n_clientes_activos + len(asegurado_only):,} clientes (+{len(asegurado_only)/n_clientes_activos:.1%})")

Verificación roles_canon:
Filas originales en roles:  275,398
Filas en roles_canon:       275,398
Nulos en id_canonico:       0

Total pólizas:          123,304
Pólizas activas (C):    50,257
Pólizas inactivas:      73,047

Sensibilidad del criterio de vigencia:
  Criterio A (solo estado=Vigente):         50,257
  Criterio B (solo fecha_fin >= corte):     54,011
  Criterio C (A + fechas, nuestro criterio): 50,257

Clientes activos totales: 35,130
Asegurados sin ser tomadores: 18,473
→ Si los incluimos: 53,603 clientes (+52.6%)


In [ ]:
# Clientes activos por ramo
# Un cliente puede aparecer en varios ramos (tiene pólizas en más de uno)
por_ramo = (
    tomadores_activos
    .groupby("ramo")["id_canonico"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_clientes")
    .reset_index()
)
por_ramo["pct_sobre_base"] = (por_ramo["n_clientes"] / n_clientes_activos * 100).round(1)

print("Clientes activos por ramo:")
print(por_ramo.to_string(index=False))

#Clientes activos por número de ramos
ramos_por_cliente = (
    tomadores_activos
    .groupby("id_canonico")["ramo"]
    .nunique()
    .rename("n_ramos")
    .reset_index()
)

dist_ramos = (
    ramos_por_cliente["n_ramos"]
    .value_counts()
    .sort_index()
    .rename("n_clientes")
    .reset_index()
)
dist_ramos.columns = ["n_ramos", "n_clientes"]
dist_ramos["pct"] = (dist_ramos["n_clientes"] / n_clientes_activos * 100).round(1)

print("\nClientes activos por número de ramos:")
print(dist_ramos.to_string(index=False))

# Resumen ejecutivo
print("\n" + "="*50)
print("RESUMEN SECCIÓN 0.1")
print("="*50)
print(f"Registros en sistema:         {n_registros_sistema:,}")
print(f"Personas únicas (dedup):      {n_personas_reales:,}  (-{n_duplicados:,})")
print(f"Tomadores con póliza activa:  {n_clientes_activos:,}")
print(f"Excluidos (asegurado-only):   {len(asegurado_only):,}")

Clientes activos por ramo:
     ramo  n_clientes  pct_sobre_base
    Salud       10566            30.1
Movilidad       10045            28.6
     Vida        8308            23.6
    Hogar        7265            20.7
 Exequias        6386            18.2
     SOAT        4105            11.7
 Mascotas        3458             9.8

Clientes activos por número de ramos:
 n_ramos  n_clientes  pct
       1       23259 66.2
       2        9121 26.0
       3        2388  6.8
       4         342  1.0
       5          20  0.1

RESUMEN SECCIÓN 0.1
Registros en sistema:         102,764
Personas únicas (dedup):      95,353  (-7,411)
Tomadores con póliza activa:  35,130
Excluidos (asegurado-only):   18,473


In [ ]:
#cargar los archivos restantes con id_canonico

#estos no tienen id_cliente  
experiencia = pd.read_csv(
    DATA_DIR / "experiencia.csv",
    parse_dates=["fecha_medicion"]
)
experiencia = experiencia.merge(
    id_canonico,
    on=["tipo_documento", "numero_documento"],
    how="left"
)

pqrs = pd.read_csv(
    DATA_DIR / "pqrs.csv",
    parse_dates=["fecha_radicacion", "fecha_respuesta", "fecha_reapertura"]
)
pqrs = pqrs.merge(
    id_canonico,
    on=["tipo_documento", "numero_documento"],
    how="left"
)

leads = pd.read_csv(
    DATA_DIR / "leads.csv",
    parse_dates=["fecha_lead"]
)
leads = leads.merge(
    id_canonico,
    on=["tipo_documento", "numero_documento"],
    how="left"
)

# estos si tienen id_cliente directo 
siniestros = pd.read_csv(
    DATA_DIR / "siniestros.csv",
    parse_dates=["fecha_ocurrencia", "fecha_aviso"]
)
siniestros = siniestros.merge(
    correspondencia[["id_cliente", "id_canonico"]],
    on="id_cliente",
    how="left"
)

campana = pd.read_csv(
    DATA_DIR / "campana_retencion_2025.csv",
    parse_dates=["fecha_asignacion"]
)
campana = campana.merge(
    correspondencia[["id_cliente", "id_canonico"]],
    on="id_cliente",
    how="left"
)

# no tienen id de persona (no necesitan mapeo)
coberturas  = pd.read_csv(DATA_DIR / "coberturas.csv")
recaudos    = pd.read_csv(DATA_DIR / "recaudos.csv",  parse_dates=["fecha_pago"])
parametros  = pd.read_csv(DATA_DIR / "parametros_financieros.csv")
asesores    = pd.read_csv(DATA_DIR / "asesores.csv",  parse_dates=["fecha_vinculacion_asesor"])

# verificar mapeos
print("Verificación de id_canonico nulos tras el mapeo:")
for nombre, df in [("siniestros", siniestros), ("campana", campana),
                   ("experiencia", experiencia), ("pqrs", pqrs), ("leads", leads)]:
    n_nulos = df["id_canonico"].isnull().sum()
    pct     = n_nulos / len(df) * 100
    print(f"  {nombre:<15} nulos en id_canonico: {n_nulos:>6,}  ({pct:.1f}%)")

archivos = {
    "clientes":   clientes,
    "polizas":    polizas,
    "roles":      roles,
    "coberturas": coberturas,
    "recaudos":   recaudos,
    "siniestros": siniestros,
    "parametros": parametros,
    "experiencia":experiencia,
    "pqrs":       pqrs,
    "asesores":   asesores,
    "campana":    campana,
    "leads":      leads
}

print(f"\n{'Archivo':<15} {'Filas':>10} {'Cols':>6} {'Cols con nulos':>15} {'% filas con algún nulo':>23}")
print("-" * 73)
for nombre, df in archivos.items():
    cols_con_nulos  = (df.isnull().sum() > 0).sum()
    pct_filas_nulas = df.isnull().any(axis=1).sum() / len(df) * 100
    print(f"{nombre:<15} {len(df):>10,} {df.shape[1]:>6} {cols_con_nulos:>15} {pct_filas_nulas:>22.1f}%")

Verificación de id_canonico nulos tras el mapeo:
  siniestros      nulos en id_canonico:      0  (0.0%)
  campana         nulos en id_canonico:      0  (0.0%)
  experiencia     nulos en id_canonico: 12,545  (12.6%)
  pqrs            nulos en id_canonico:  4,054  (8.0%)
  leads           nulos en id_canonico: 20,688  (69.3%)

Archivo              Filas   Cols  Cols con nulos  % filas con algún nulo
-------------------------------------------------------------------------
clientes           102,764     17               5                   70.1%
polizas            123,304     11               3                   84.4%
roles              275,398      3               0                    0.0%
coberturas         445,715      3               1                   10.2%
recaudos         2,018,014      4               0                    0.0%
siniestros         110,561     10               1                    1.8%
parametros              32      4               0                    0.0%
experie

In [35]:
#Nulos detallados por archivo
print("NULOS DETALLADOS POR ARCHIVO")
print("="*60)
for nombre, df in archivos.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f"\n{nombre}:")
        for col, n in nulos.items():
            print(f"  {col:<35} {n:>8,}  ({n/len(df)*100:.1f}%)")



NULOS DETALLADOS POR ARCHIVO

clientes:
  estado_civil                          14,411  (14.0%)
  ocupacion                              8,894  (8.7%)
  ingreso_estimado                      40,664  (39.6%)
  num_dependientes_mayores              20,583  (20.0%)
  num_dependientes_menores              20,571  (20.0%)

polizas:
  codigo_asesor                         77,068  (62.5%)
  fecha_cancelacion                     50,481  (40.9%)
  motivo_cancelacion                    64,034  (51.9%)

coberturas:
  valor_asegurado                       45,676  (10.2%)

siniestros:
  valor_pagado                           1,971  (1.8%)

experiencia:
  tipo_documento                         8,072  (8.1%)
  numero_documento                       8,072  (8.1%)
  nps                                    3,947  (4.0%)
  comentario                             1,715  (1.7%)
  id_canonico                           12,545  (12.6%)

pqrs:
  fecha_respuesta                          270  (0.5%)
  fecha_reaper

In [36]:
# Integridad referencial: pólizas en roles que no existen en polizas
polizas_en_roles   = set(roles["codigo_poliza"].unique())
polizas_en_polizas = set(polizas["codigo_poliza"].unique())
huerfanos_roles    = polizas_en_roles - polizas_en_polizas

print(f"\nPólizas en roles sin registro en polizas.csv: {len(huerfanos_roles):,}")

# Integridad referencial: pólizas en siniestros que no existen en polizas
polizas_en_siniestros = set(siniestros["codigo_poliza"].unique())
huerfanos_siniestros  = polizas_en_siniestros - polizas_en_polizas

print(f"Pólizas en siniestros sin registro en polizas.csv: {len(huerfanos_siniestros):,}")

# Integridad referencial: pólizas en recaudos que no existen en polizas
recaudos["codigo_poliza"] = recaudos["codigo_poliza"].astype(str)
polizas_en_recaudos  = set(recaudos["codigo_poliza"].unique())
huerfanos_recaudos   = polizas_en_recaudos - polizas_en_polizas

print(f"Pólizas en recaudos sin registro en polizas.csv:   {len(huerfanos_recaudos):,}")

# Campana: confirmar origen de los nulos
print("\nCampaña — nulos por grupo:")
print(campana.groupby("grupo")[["tipo_accion", "costo_accion"]].apply(
    lambda x: x.isnull().sum()
))

#Polizas: 13,553 con fecha_cancelacion pero sin motivo
sin_motivo = polizas[
    polizas["fecha_cancelacion"].notnull() &
    polizas["motivo_cancelacion"].isnull()
]
print(f"\nPólizas canceladas sin motivo registrado: {len(sin_motivo):,}")
print("Distribución por ramo:")
print(sin_motivo["ramo"].value_counts())


Pólizas en roles sin registro en polizas.csv: 0
Pólizas en siniestros sin registro en polizas.csv: 34,025
Pólizas en recaudos sin registro en polizas.csv:   0

Campaña — nulos por grupo:
             tipo_accion  costo_accion
grupo                                 
Control             4959          4959
Tratamiento            0          3136

Pólizas canceladas sin motivo registrado: 14,654
Distribución por ramo:
ramo
SOAT         13845
Salud          196
Movilidad      171
Vida           159
Hogar          115
Exequias       102
Mascotas        66
Name: count, dtype: int64


In [38]:
#Caracterización MCAR / MAR / MNAR
# Variables críticas para modelos: ingreso_estimado, estado_civil,
# ocupacion, num_dependientes_mayores, num_dependientes_menores
# Crear tabla base de trabajo: solo clientes activos con sus atributos
clientes_canon = clientes[clientes["id_cliente"].isin(
    correspondencia[correspondencia["id_cliente"] == correspondencia["id_canonico"]]["id_cliente"]
)].copy()

# Indicadores de missingness
clientes_canon["miss_ingreso"]      = clientes_canon["ingreso_estimado"].isnull().astype(int)
clientes_canon["miss_estado_civil"] = clientes_canon["estado_civil"].isnull().astype(int)
clientes_canon["miss_ocupacion"]    = clientes_canon["ocupacion"].isnull().astype(int)
clientes_canon["miss_dependientes"] = clientes_canon["num_dependientes_mayores"].isnull().astype(int)

# ingreso_estimado: ¿el nulo varía por estrato?
# Si MCAR → tasa de nulos igual en todos los estratos
# Si MAR  → tasa varía sistemáticamente por estrato (observable)
# Si MNAR → varía pero los que tienen datos bajos son los que más falta
print("Tasa de nulos en ingreso_estimado por estrato:")
tasa_ingreso_estrato = (
    clientes_canon.groupby("estrato", observed=True)["miss_ingreso"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "tasa_nulo", "count": "n"})
)
tasa_ingreso_estrato["tasa_nulo"] = (tasa_ingreso_estrato["tasa_nulo"] * 100).round(1)
print(tasa_ingreso_estrato)

# ingreso_estimado: ¿varía por canal de adquisición (ramo)?
# El ramo puede indicar cómo entró el cliente: si entró por SOAT puede
# que no se haya capturado el ingreso
ramos_cliente = (
    tomadores_activos
    .groupby("id_canonico")["ramo"]
    .apply(lambda x: x.iloc[0])  # ramo principal (primero)
    .reset_index()
)
clientes_ramo = clientes_canon.merge(
    correspondencia[["id_cliente", "id_canonico"]],
    left_on="id_cliente", right_on="id_cliente", how="left"
).merge(ramos_cliente, on="id_canonico", how="left")

print("\nTasa de nulos en ingreso_estimado por ramo principal:")
tasa_ingreso_ramo = (
    clientes_ramo.groupby("ramo")["miss_ingreso"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "tasa_nulo", "count": "n"})
    .sort_values("tasa_nulo", ascending=False)
)
tasa_ingreso_ramo["tasa_nulo"] = (tasa_ingreso_ramo["tasa_nulo"] * 100).round(1)
print(tasa_ingreso_ramo)



Tasa de nulos en ingreso_estimado por estrato:
         tasa_nulo      n
estrato                  
1             34.9   8879
2             36.0  14084
3             36.9  18438
4             38.6  21617
5             38.7  19893
6             36.6  12442

Tasa de nulos en ingreso_estimado por ramo principal:
           tasa_nulo     n
ramo                      
SOAT            35.6  3313
Mascotas        31.1  2679
Vida            28.1  5478
Movilidad       26.9  6833
Hogar           25.7  4724
Salud           25.0  7183
Exequias        23.6  4920


In [39]:
#  estado_civil: ¿varía por estrato y edad?
print("\nTasa de nulos en estado_civil por estrato:")
tasa_ec_estrato = (
    clientes_canon.groupby("estrato", observed=True)["miss_estado_civil"]
    .mean().mul(100).round(1).rename("tasa_nulo_%")
)
print(tasa_ec_estrato)

# num_dependientes: comparar distribución de otras variables
# entre quienes SI tienen el dato vs quienes NO lo tienen
# Si MCAR → no hay diferencia sistemática
print("\nComparación por grupo de missingness en num_dependientes:")
comp = clientes_canon.groupby("miss_dependientes").agg(
    n                = ("id_cliente", "count"),
    estrato_modal    = ("estrato",    lambda x: x.mode()[0] if len(x.mode()) > 0 else None),
    pct_sin_ingreso  = ("miss_ingreso",    "mean"),
    pct_sin_ec       = ("miss_estado_civil", "mean"),
).round(3)
comp["pct_sin_ingreso"] = (comp["pct_sin_ingreso"] * 100).round(1)
comp["pct_sin_ec"]      = (comp["pct_sin_ec"]      * 100).round(1)
print(comp)

# Test Little's MCAR (si scipy disponible)
# Una aproximación práctica: correlación entre indicadores de missingness
miss_vars = ["miss_ingreso", "miss_estado_civil", "miss_ocupacion", "miss_dependientes"]
print("\nCorrelación entre indicadores de missingness:")
print(clientes_canon[miss_vars].corr().round(3))
# Si correlaciones altas → los nulos no son independientes → no MCAR

# Siniestros huérfanos: caracterización
siniestros_huerfanos = siniestros[~siniestros["codigo_poliza"].isin(polizas["codigo_poliza"])]
print(f"\nSiniestros huérfanos: {len(siniestros_huerfanos):,}")
print("Rango de fechas de ocurrencia:")
print(f"  Min: {siniestros_huerfanos['fecha_ocurrencia'].min()}")
print(f"  Max: {siniestros_huerfanos['fecha_ocurrencia'].max()}")
print("¿Los siniestros huérfanos son más antiguos que los emparejados?")
print(f"  Fecha media huérfanos:   {siniestros_huerfanos['fecha_ocurrencia'].mean().date()}")
siniestros_ok = siniestros[siniestros["codigo_poliza"].isin(polizas["codigo_poliza"])]
print(f"  Fecha media emparejados: {siniestros_ok['fecha_ocurrencia'].mean().date()}")


Tasa de nulos en estado_civil por estrato:
estrato
1    14.5
2    14.1
3    14.2
4    14.0
5    13.9
6    13.8
Name: tasa_nulo_%, dtype: float64

Comparación por grupo de missingness en num_dependientes:
                       n estrato_modal  pct_sin_ingreso  pct_sin_ec
miss_dependientes                                                  
0                  76187             4             37.2        14.0
1                  19166             4             37.7        14.5

Correlación entre indicadores de missingness:
                   miss_ingreso  miss_estado_civil  miss_ocupacion  \
miss_ingreso              1.000             -0.001          -0.067   
miss_estado_civil        -0.001              1.000           0.004   
miss_ocupacion           -0.067              0.004           1.000   
miss_dependientes         0.004              0.006          -0.003   

                   miss_dependientes  
miss_ingreso                   0.004  
miss_estado_civil              0.006  
miss_ocu

In [40]:
#Verificar tipos actuales debido a que no hizo match
print("Tipos de codigo_poliza:")
print(f"  polizas:    {polizas['codigo_poliza'].dtype}")
print(f"  siniestros: {siniestros['codigo_poliza'].dtype}")



Tipos de codigo_poliza:
  polizas:    str
  siniestros: int64


In [41]:
#Corregir tipo y rehacerIntegridad referencial
siniestros["codigo_poliza"] = siniestros["codigo_poliza"].astype(str)

polizas_en_siniestros_ok = set(siniestros["codigo_poliza"].unique())
polizas_en_polizas_ok    = set(polizas["codigo_poliza"].unique())
huerfanos_real           = polizas_en_siniestros_ok - polizas_en_polizas_ok

print(f"\nSiniestros — huérfanos REALES (tipos corregidos): {len(huerfanos_real):,}")
print(f"Total siniestros: {len(siniestros):,}")
print(f"Pct huérfanos: {len(huerfanos_real)/len(siniestros)*100:.1f}%")



Siniestros — huérfanos REALES (tipos corregidos): 0
Total siniestros: 110,561
Pct huérfanos: 0.0%


In [42]:
#Análisis de ocupacion
print("Tasa de nulos en ocupacion por estrato:")
tasa_ocup = (
    clientes_canon.groupby("estrato", observed=True)["miss_ocupacion"]
    .mean().mul(100).round(1).rename("tasa_nulo_%")
)
print(tasa_ocup)

print("\nOcupacion: diferencia entre quien tiene y no tiene el dato")
comp_ocup = clientes_canon.groupby("miss_ocupacion").agg(
    n               = ("id_cliente", "count"),
    estrato_medio   = ("estrato", lambda x: x.cat.codes.mean().round(2)),
    pct_sin_ingreso = ("miss_ingreso", "mean"),
).round(3)
comp_ocup["pct_sin_ingreso"] = (comp_ocup["pct_sin_ingreso"]*100).round(1)
print(comp_ocup)

Tasa de nulos en ocupacion por estrato:
estrato
1    10.0
2     9.3
3     8.7
4     8.4
5     8.0
6     8.6
Name: tasa_nulo_%, dtype: float64

Ocupacion: diferencia entre quien tiene y no tiene el dato
                    n  estrato_medio  pct_sin_ingreso
miss_ocupacion                                       
0               87089           2.71             38.3
1                8264           2.61             26.8


# Sección 0.2 — Auditoría de Calidad de los Doce Archivos

## Objetivo
Identificar todos los problemas de calidad en los archivos fuente, reportar su magnitud,
caracterizar el mecanismo que los explica, documentar la decisión tomada y estimar el
efecto sobre los resultados de las secciones siguientes.


## 1. Panorama general de completitud

| Archivo | Filas | Columnas | Filas con algún nulo | % afectado |
|---|---|---|---|---|
| `clientes.csv` | 102,764 | — | ~72,000 | 70.1% |
| `polizas.csv` | 123,352 | — | ~104,100 | 84.4% |
| `roles_poliza.csv` | 182,847 | — | 0 | 0.0% |
| `coberturas.csv` | — | — | ~45,700 | 10.2% |
| `recaudos.csv` | — | — | 0 | 0.0% |
| `siniestros.csv` | 110,561 | — | ~2,000 | 1.8% |
| `parametros_financieros.csv` | — | — | 0 | 0.0% |
| `experiencia.csv` | — | — | ~17,500 | 17.5% |
| `pqrs.csv` | — | — | ~mayoría | 93.4% |
| `asesores.csv` | — | — | 0 | 0.0% |
| `campana_retencion_2025.csv` | — | — | ~8,100 | 35.4% |
| `leads.csv` | — | — | ~20,700 | 69.3% |


## 2. Integridad referencial

Se verificó que todas las llaves foráneas resuelvan correctamente contra sus tablas maestras.

| Relación verificada | Huérfanos encontrados | Observación |
|---|---|---|
| `roles_poliza.codigo_poliza` → `polizas.codigo_poliza` | **0** | ✅ Perfecta |
| `siniestros.codigo_poliza` → `polizas.codigo_poliza` | **0** | ✅ Perfecta (ver nota) |
| `recaudos.codigo_poliza` → `polizas.codigo_poliza` | **0** | ✅ Perfecta |

> **Nota técnica:** El chequeo inicial reportó 34,025 pólizas huérfanas en siniestros.
> Se identificó como un artefacto de tipo de dato: `polizas.codigo_poliza` había sido
> convertido a `str` mientras `siniestros.codigo_poliza` permanecía como `int64`.
> Al unificar tipos, los huérfanos reales son **cero**. Corrección aplicada:
> `siniestros["codigo_poliza"] = siniestros["codigo_poliza"].astype(str)`

## 3. Nulos por variable — detalle y mecanismo

### 3.1 `clientes.csv`

#### `ingreso_estimado` — 40,664 nulos (39.6%)

**Magnitud:** crítica. Afecta a 2 de cada 5 clientes.

**Evidencia empírica del mecanismo:**

Tasa de nulos por ramo principal del cliente:

| Ramo | Tasa nulos % |
|---|---|
| SOAT | 35.6 |
| Mascotas | 31.1 |
| Vida | 28.1 |
| Movilidad | 26.9 |
| Hogar | 25.7 |
| Salud | 25.0 |
| Exequias | 23.6 |

Tasa de nulos por estrato (mínima variación: 34.9% a 38.7%):

| Estrato | Tasa nulos % |
|---|---|
| 1 | 34.9 |
| 2 | 36.0 |
| 3 | 36.9 |
| 4 | 38.6 |
| 5 | 38.7 |
| 6 | 36.6 |

**Mecanismo: MAR (Missing At Random)**
La tasa varía 12 pp según el ramo/canal de adquisición (23.6% Exequias vs 35.6% SOAT).
Los productos transaccionales como SOAT se contratan sin un proceso de vinculación comercial
donde el asesor captura el perfil socioeconómico. Por estrato, la variación es mínima (<4 pp),
lo que descarta MNAR fuerte (si fuera MNAR, estratos bajos mostrarían tasas
significativamente mayores por renuencia a declarar ingresos informales).

**Decisión:** imputar con mediana por grupo `(ramo × estrato)` para los modelos de CLV.
En el modelo de churn añadir un indicador binario `ingreso_faltante` como feature
independiente, ya que la ausencia del dato es en sí informativa del canal de adquisición.

**Efecto:** sin imputación, el CLV promedio de clientes SOAT quedaría sesgado a la baja.
La variable `ingreso_faltante` puede aparecer como feature relevante en el modelo de churn.


#### `estado_civil` — 14,411 nulos (14.0%)

**Mecanismo: MCAR (Missing Completely At Random)**

Tasa prácticamente uniforme entre estratos (13.8%–14.5%, rango de 0.7 pp).
Correlación del indicador de missingness con todos los demás indicadores ≈ 0 (máximo: 0.006).
Los nulos no dependen de ninguna variable observable.

**Decisión:** imputar con moda por grupo de edad para modelos que requieran la variable.

**Efecto:** sesgo de imputación mínimo dado el mecanismo MCAR.


#### `num_dependientes_mayores` / `num_dependientes_menores` — ~20,580 nulos (20.0%)

**Mecanismo: MCAR**

Comparación entre clientes con y sin el dato:

| Grupo | n | % sin ingreso | % sin estado civil |
|---|---|---|---|
| Con dato (miss=0) | 76,187 | 37.2% | 14.0% |
| Sin dato (miss=1) | 19,166 | 37.7% | 14.5% |

Diferencia de apenas 0.5 pp en ambas variables de control. Los grupos son estadísticamente
indistinguibles en las variables observables. Correlación con otros indicadores de missingness < 0.01.

**Decisión:** imputar con mediana por estrato.

**Efecto:** sesgo negligible dado el mecanismo MCAR.


#### `ocupacion` — 8,894 nulos (8.7%)

**Mecanismo: MAR moderado**

Tasa varía levemente por estrato (8.0%–10.0%). Hallazgo relevante: los clientes sin
`ocupacion` tienen *menor* tasa de faltantes en `ingreso_estimado` (26.8% vs 38.3%).
Correlación negativa entre indicadores de missingness = -0.067.

Interpretación: existen dos patrones de captura según canal. Los canales presenciales
(con asesor) capturan ocupación pero no siempre ingreso; los canales digitales/transaccionales
capturan ingreso declarado pero no ocupación. El mecanismo depende del canal, variable
observable a través de `ramo` y `codigo_asesor`.

**Decisión:** imputar con moda por `(estrato × ramo)`.

**Efecto:** bajo impacto en modelos; `ocupacion` es variable de segunda prioridad en churn.


### 3.2 `polizas.csv`

#### `codigo_asesor` — 77,068 nulos (62.5%)

**Mecanismo: Estructural**
Los canales sin asesor asignado (venta digital, SOAT masivo, renovación automática)
no generan un registro de asesor. El nulo indica el canal, no un error de captura.

**Decisión:** conservar nulo como categoría explícita `"sin_asesor"` en modelos.
Es una variable proxy del canal de adquisición.

**Efecto:** en la Sección 5, el canal es una variable confusora relevante en el
análisis de la campaña de retención.


#### `fecha_cancelacion` — 50,481 nulos (40.9%) / `motivo_cancelacion` — 64,034 nulos (51.9%)

**Mecanismo: Estructural (parcial) + MNAR (parcial)**

- Nulos en `fecha_cancelacion`: corresponden a las 50,257 pólizas vigentes al corte.
  El nulo significa "no cancelada", no "dato faltante".
- Exceso de nulos en `motivo_cancelacion` sobre `fecha_cancelacion` (13,777 adicionales):
  pólizas canceladas sin motivo registrado. De estas, 13,845 corresponden a SOAT,
  ramo para el cual el sistema no solicita motivo (cancelación automática al vencimiento).

  Distribución de pólizas canceladas sin motivo:

  | Ramo | Sin motivo |
  |---|---|
  | SOAT | 13,845 |
  | Salud | 196 |
  | Movilidad | 171 |
  | Vida | 159 |
  | Hogar | 115 |
  | Exequias | 102 |
  | Mascotas | 66 |

**Decisión:** para SOAT, excluir `motivo_cancelacion` del análisis de churn.
Para los demás ramos, tratar el motivo faltante en canceladas como categoría
"motivo no registrado" (posible MNAR si clientes que se van por precio evitan declararlo).

**Efecto:** la variable target de churn (Sección 2) se construirá desde `estado` y
`fecha_fin_vigencia`, no desde `motivo_cancelacion`.


### 3.3 `coberturas.csv`

#### `valor_asegurado` — 45,676 nulos (10.2%)

**Mecanismo: Estructural**
Coberturas de tipo ilimitado no tienen un tope máximo definido. El nulo no es
ausencia de información sino la representación de "sin límite".

**Decisión:** crear indicador binario `cobertura_ilimitada = 1` donde `valor_asegurado`
es nulo. En modelos de CLV usar este indicador en lugar del valor numérico.

**Efecto:** sin esta corrección, se imputaría erróneamente un tope a coberturas ilimitadas.


### 3.4 `siniestros.csv`

#### `valor_pagado` — 1,971 nulos (1.8%)

**Mecanismo: MAR**
Los siniestros abiertos (en proceso de liquidación) no tienen valor pagado final.
El nulo depende del estado del siniestro, variable observable.

**Decisión:** excluir siniestros sin `valor_pagado` del cálculo de costo esperado
acumulado. Usar `valor_reclamado` como proxy cuando el modelo lo requiera.

**Efecto:** subestimación marginal del costo histórico en ~1.8% de los siniestros.


### 3.5 `experiencia.csv`

#### `tipo_documento` / `numero_documento` — 8,072 nulos (8.1%) / `nps` — 3,947 nulos (4.0%)

**Mecanismo: MAR**
Encuestas respondidas sin identificación (modo anónimo) o por personas cuyo documento
no coincide con ningún cliente registrado (ex-clientes, terceros asegurados).
El 12.6% de respuestas no pudo vincularse a ningún `id_canonico`.

**Decisión:** excluir registros sin `id_canonico` del análisis individual de experiencia.
Incluirlos en el análisis agregado por ramo de la Sección 4, ya que el contenido
textual es válido independientemente de la identidad.

**Efecto:** el universo analizable de NLP en la Sección 4 se reduce en 12.6%.


### 3.6 `pqrs.csv`

#### Nulos en `fecha_respuesta` y `fecha_reapertura` — 93.4% de filas con algún nulo

**Mecanismo: Estructural**
- `fecha_respuesta`: nulo cuando el caso está abierto (aún sin resolución).
- `fecha_reapertura`: nulo cuando el caso no fue reabierto (mayoría de los casos).

Ambos nulos son esperados por diseño del proceso, no errores de captura.

**Decisión:** no imputar. Usar indicadores derivados: `caso_cerrado = fecha_respuesta.notnull()`,
`caso_reabierto = fecha_reapertura.notnull()`.


### 3.7 `campana_retencion_2025.csv`

#### `tipo_accion` / `costo_accion` — 35.4% nulos

**Mecanismo:** doble naturaleza:

| Grupo | Nulos en tipo_accion | Nulos en costo_accion | Interpretación |
|---|---|---|---|
| Control | 4,959 (100%) | 4,959 (100%) | Estructural: sin acción por diseño |
| Tratamiento | 0 (0%) | 3,136 (35.4%) | MAR: acción sin costo unitario registrado |

Los nulos en tratamiento corresponden a acciones (llamadas telefónicas, visitas)
donde el sistema no registra un costo unitario predefinido.

**Decisión:** grupo control: nulo = sin intervención (variable indicadora).
Tratamiento sin costo: imputar con mediana por `tipo_accion`.

**Efecto:** en la Sección 5, el costo imputable permite calcular el ROI de la campaña.


### 3.8 `leads.csv`

#### `id_canonico` — 20,688 nulos (69.3%)

**Mecanismo: Estructural**
La mayoría de leads son prospectos que nunca compraron una póliza. No tienen
`id_canonico` porque no existen en `clientes.csv`. Este es el comportamiento
esperado en un funnel de conversión.

**Decisión:** usar solo los leads con `id_canonico` para análisis de conversión
(leads que se convirtieron en clientes). Los leads sin `id_canonico` son parte
del denominador del embudo, no del análisis de cliente activo.


## 4. Resumen de decisiones

| Variable | Mecanismo | Decisión de tratamiento |
|---|---|---|
| `ingreso_estimado` | MAR | Imputar mediana por `(ramo × estrato)` + feature `ingreso_faltante` |
| `estado_civil` | MCAR | Imputar moda por grupo de edad |
| `num_dependientes` | MCAR | Imputar mediana por estrato |
| `ocupacion` | MAR moderado | Imputar moda por `(estrato × ramo)` |
| `codigo_asesor` | Estructural | Categoría explícita `"sin_asesor"` |
| `fecha_cancelacion` | Estructural | Nulo = vigente |
| `motivo_cancelacion` SOAT | Estructural | Excluir del análisis de churn SOAT |
| `valor_asegurado` | Estructural | Indicador binario `cobertura_ilimitada` |
| `valor_pagado` siniestros | MAR | Excluir; usar `valor_reclamado` como proxy |
| `id_canonico` en experiencia | MAR | Excluir del análisis individual; incluir en NLP |
| `tipo_accion` / `costo_accion` Control | Estructural | Indicador de no-intervención |
| `costo_accion` Tratamiento | MAR | Imputar mediana por `tipo_accion` |